# Vimeo-90K chunked training on Colab

Automates: download one ~6-10GB Vimeo-90K chunk from Kaggle (`wangsally/vimeo-90k-1` .. `-10`) -> train `BaselineAutoencoder` on it -> checkpoint to Google Drive -> delete the chunk -> move to the next chunk. Repeats for every chunk listed in `CHUNK_NUMBERS`.

**Before running:**
- A Kaggle account + API token (kaggle.com/settings -> API -> Create New Token). You upload it once; it is then cached on your Drive so future reconnects do not ask again.
- A Google account for Drive (checkpoints and progress live there so a disconnect does not lose anything).

**If the session disconnects:** reopen this notebook and Runtime -> Run all again. Already-downloaded/trained chunks are recorded in `progress.json` on Drive and are skipped automatically; training resumes from the last saved checkpoint.

**At the end:** the last cell downloads `best.pt` straight to your machine.

**Design notes (full reasoning in the chat this notebook came from):**
- Frames are never copied off the Kaggle chunk into a second location - each chunk's folders are symlinked directly into `data/external/vimeo_septuplet/sequences/`, so local disk use stays close to one chunk's size (~6-10GB) at a time, never the full ~89GB.
- The train/test split written for each chunk (`sep_trainlist.txt` / `sep_testlist.txt`) is a fresh, reproducible split *of that chunk only*. It exists purely to give training a validation signal per chunk, and is discarded when the chunk is deleted. It is **not** the official Vimeo-90K split, and it is **not** meant to be a final benchmark.
- To compare this model against one trained on the full dataset elsewhere (e.g. a friend's machine), evaluate both on a fixed, independent set neither model trained on - DAVIS's existing test split (already in this repo, untouched by anything here) is a good candidate for that.

In [ ]:
from pathlib import Path

# --- Which Kaggle chunks to cycle through (wangsally/vimeo-90k-1 .. vimeo-90k-10) ---
CHUNK_NUMBERS = list(range(1, 11))  # e.g. [1, 2, 3] to do fewer if you are short on time/disk

# --- Training ---
EPOCHS_PER_CHUNK = 2     # epochs run on each chunk before moving to the next
BATCH_SIZE = 32
LATENT_CHANNELS = None   # None -> use configs/default.json's value. Keep this THE SAME across every
                          # run you resume from - changing it will break loading an existing checkpoint.
LEARNING_RATE = None     # None -> use configs/default.json's value
CROP_SIZE = 256          # Vimeo frames are 448x256 natively; crop size must be divisible by 16
SEED = 42
NUM_WORKERS = 2          # Colab is Linux (fork-based) - safe to use >0, unlike the Windows default of 0

# --- Where things live ---
REPO_URL = 'https://github.com/yuvidewan/neural_streaming.git'
REPO_DIR = Path('/content/neural_streaming')
DRIVE_PROJECT_DIR = Path('/content/drive/MyDrive/neural_streaming_colab')
LOCAL_SCRATCH_DIR = Path('/content/vimeo_scratch')  # wiped and recreated per chunk, never accumulates

DELETE_CHUNK_AFTER_TRAINING = True
KAGGLE_DATASET_OWNER = 'wangsally'
KAGGLE_DATASET_PREFIX = 'vimeo-90k'  # -> wangsally/vimeo-90k-1, vimeo-90k-2, ...

# --- Milestone 8A: quantization-aware training (distortion-only noise relaxation) ---
# See src/nvc/training/quantization_noise.py for the mechanism itself. Flip
# QAT_ENABLED to True to run the QAT experiment instead of baseline training;
# everything above (dataset, batch size, crop size, optimizer, seed, chunk
# schedule) stays identical between the two runs on purpose, so the only
# difference is this noise relaxation.
QAT_ENABLED = False        # False = ordinary baseline training (unchanged behavior)
QAT_BITS = 4                # bit depth the noise relaxation targets
QAT_MODE = 'per_channel'    # must match QAT_CALIBRATION_PATH's own recorded mode

# Frozen calibration artifact (Vimeo TRAIN split only) supplying the training-
# time noise scale. Generate it BEFORE flipping QAT_ENABLED on, from the
# checkpoint named in QAT_RESUME_FROM below, e.g. (run locally or in a cell
# here after Section 3 installs the package):
#   python scripts/calibrate_quantizer.py --checkpoint <QAT_RESUME_FROM> \
#       --manifest <a Vimeo train manifest> --bits 4 --mode per_channel \
#       --output <this path>
QAT_CALIBRATION_PATH = DRIVE_PROJECT_DIR / 'calibration' / 'vimeo_qat_4bit_train.json'

# Checkpoint to fine-tune from when starting the QAT experiment fresh (only
# used the first time - once the QAT checkpoint dir has its own latest.pt,
# resume continues from there instead). Point this at the existing
# non-QAT Vimeo checkpoint (Drive's checkpoints/best.pt - locally saved as
# vimeo_epoch17_best.pt) so QAT is a controlled continuation, not a
# from-scratch retrain.
QAT_RESUME_FROM = DRIVE_PROJECT_DIR / 'checkpoints' / 'best.pt'


# --- Milestone 8A control run: isolates "the noise relaxation helped" from
# "more training helped." Fine-tunes from the SAME QAT_RESUME_FROM checkpoint,
# for the same epoch budget, with plain MSE and NO noise - a same-budget
# comparison point for the QAT run. Mutually exclusive with QAT_ENABLED (only
# one of the two should be True in a given run).
QAT_CONTROL_RUN = False    # True = matched-budget plain-MSE fine-tune, own checkpoint dir
assert not (QAT_ENABLED and QAT_CONTROL_RUN), 'QAT_ENABLED and QAT_CONTROL_RUN are mutually exclusive'


## 1. Mount Google Drive

Checkpoints and progress state are written here, not to Colab's local disk, so nothing is lost on disconnect.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_PROJECT_DIR.mkdir(parents=True, exist_ok=True)
# Milestone 8A: each experiment type gets its own checkpoint dir and its
# own chunk-completion progress file, distinct from the others - otherwise
# a run would inherit 'already completed' chunks from a different run's
# progress.json and train on nothing, or worse, overwrite that run's
# checkpoints in place.
if QAT_ENABLED:
    RUN_TYPE = 'qat'
elif QAT_CONTROL_RUN:
    RUN_TYPE = 'qat_control'
else:
    RUN_TYPE = 'baseline'
_CHECKPOINT_SUBDIR = {'baseline': 'checkpoints', 'qat': 'checkpoints_qat_noise', 'qat_control': 'checkpoints_qat_control'}
_PROGRESS_NAME = {'baseline': 'progress.json', 'qat': 'progress_qat_noise.json', 'qat_control': 'progress_qat_control.json'}
CHECKPOINT_DIR = DRIVE_PROJECT_DIR / _CHECKPOINT_SUBDIR[RUN_TYPE]
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
PROGRESS_PATH = DRIVE_PROJECT_DIR / _PROGRESS_NAME[RUN_TYPE]
print(f'Run type: {RUN_TYPE}')
print(f'Drive project dir: {DRIVE_PROJECT_DIR}')
print(f'Checkpoints will be saved to: {CHECKPOINT_DIR}')

## 2. Kaggle API credentials

Uploaded once via the file picker below, then cached on Drive - future reconnects reuse the cached copy automatically and will not prompt again.

In [ ]:
import os
import shutil
from pathlib import Path

kaggle_dir = Path.home() / '.kaggle'
kaggle_dir.mkdir(parents=True, exist_ok=True)
kaggle_json_target = kaggle_dir / 'kaggle.json'
drive_kaggle_json = DRIVE_PROJECT_DIR / 'kaggle.json'

if drive_kaggle_json.is_file():
    shutil.copy(drive_kaggle_json, kaggle_json_target)
    print(f'Reused Kaggle API token from {drive_kaggle_json}')
else:
    print('No saved Kaggle token found on Drive - upload your kaggle.json now.')
    print('(Get it from https://www.kaggle.com/settings -> API -> Create New Token)')
    from google.colab import files
    uploaded = files.upload()
    uploaded_name = next(iter(uploaded))
    shutil.move(uploaded_name, kaggle_json_target)
    shutil.copy(kaggle_json_target, drive_kaggle_json)
    print(f'Saved a copy to {drive_kaggle_json} - future reconnects will reuse it automatically.')

os.chmod(kaggle_json_target, 0o600)

## 3. Get the project code

Clones `neural_streaming` from GitHub and installs it editable, so `import nvc` uses the exact same code as your (and your collaborator's) local checkouts.

In [ ]:
import os
import subprocess

if REPO_DIR.is_dir():
    print(f'{REPO_DIR} already exists - pulling latest')
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull'], check=True)
else:
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
print(f'cwd: {os.getcwd()}')

In [ ]:
!pip install -q -r requirements.txt
!pip install -e .
!pip install -q kaggle

In [ ]:
# Fails loudly right here if the editable install above didn't actually take -
# better than a confusing ModuleNotFoundError several cells later.
import nvc
print(f'nvc package loaded OK from: {nvc.__file__}')

## 4. Imports and setup

In [ ]:
import json
import random
import shutil
import subprocess
import zipfile
from pathlib import Path

import torch

from nvc.data.loaders import create_sequence_test_loader, create_sequence_train_loader
from nvc.data.vimeo import build_sequence_manifest
from nvc.models import BaselineAutoencoder
from nvc.training import (
    QuantizationNoise,
    resume_training_state,
    save_checkpoint,
    train_one_epoch,
    validate_one_epoch,
)
from nvc.utils.config import load_default_config
from nvc.utils.device import get_device
from nvc.utils.seed import seed_everything

defaults = load_default_config()
device = get_device()
seed_everything(SEED)

VIMEO_ROOT = defaults.vimeo_root
VIMEO_ROOT.mkdir(parents=True, exist_ok=True)

latent_channels = LATENT_CHANNELS or defaults.latent_channels
learning_rate = LEARNING_RATE or defaults.learning_rate
print(f'VIMEO_ROOT: {VIMEO_ROOT}')
print(f'latent_channels={latent_channels}  learning_rate={learning_rate}')

## 5. Helper functions

Chunk download/extraction, zero-copy symlink merge into `VIMEO_ROOT/sequences`, per-chunk split-list generation, and manifest building via the existing `nvc.data.vimeo` pipeline.

In [ ]:
def _find_sequences_source_root(chunk_dir: Path) -> Path:
    '''Locate the folder whose children are Vimeo "<group>" directories, by
    finding the first im1.png anywhere under chunk_dir and walking up two
    levels (im1.png -> <clip>/ -> <group>/ -> the folder holding all groups).
    '''
    for im1 in chunk_dir.rglob('im1.png'):
        clip_dir = im1.parent
        group_dir = clip_dir.parent
        return group_dir.parent
    raise RuntimeError(f'No im1.png found anywhere under {chunk_dir} - unexpected chunk layout')


def download_and_extract_chunk(chunk_number: int, scratch_dir: Path) -> Path:
    '''Download one wangsally/vimeo-90k-N Kaggle dataset and extract it.
    Returns the folder whose direct children are Vimeo "<group>" folders.
    '''
    if scratch_dir.exists():
        shutil.rmtree(scratch_dir)
    scratch_dir.mkdir(parents=True)

    slug = f'{KAGGLE_DATASET_OWNER}/{KAGGLE_DATASET_PREFIX}-{chunk_number}'
    print(f'[chunk {chunk_number}] downloading {slug} ...')
    subprocess.run(['kaggle', 'datasets', 'download', '-d', slug, '-p', str(scratch_dir)], check=True)

    zips = list(scratch_dir.glob('*.zip'))
    if not zips:
        raise RuntimeError(f'[chunk {chunk_number}] no .zip downloaded into {scratch_dir}')
    print(f'[chunk {chunk_number}] extracting {zips[0].name} ...')
    with zipfile.ZipFile(zips[0]) as zf:
        zf.extractall(scratch_dir)
    zips[0].unlink()

    return _find_sequences_source_root(scratch_dir)


def relink_sequences_to_chunk(chunk_group_root: Path) -> list:
    '''Point VIMEO_ROOT/sequences at exactly this chunk's group folders via
    symlinks - no copying, so this costs no extra disk beyond the chunk
    itself. Returns the list of group directory names now available.
    '''
    sequences_dir = VIMEO_ROOT / 'sequences'
    if sequences_dir.exists() or sequences_dir.is_symlink():
        shutil.rmtree(sequences_dir, ignore_errors=True)
    sequences_dir.mkdir(parents=True)

    group_names = []
    for group_dir in sorted(p for p in chunk_group_root.iterdir() if p.is_dir()):
        (sequences_dir / group_dir.name).symlink_to(group_dir, target_is_directory=True)
        group_names.append(group_dir.name)
    return group_names


def discover_complete_sequence_ids(sequences_dir: Path) -> list:
    '''List "<group>/<clip>" ids that have all 7 im*.png frames present.'''
    ids = []
    for group_dir in sorted(p for p in sequences_dir.iterdir() if p.is_dir()):
        for clip_dir in sorted(p for p in group_dir.iterdir() if p.is_dir()):
            if all((clip_dir / f'im{i}.png').is_file() for i in range(1, 8)):
                ids.append(f'{group_dir.name}/{clip_dir.name}')
    return ids


def write_chunk_split_lists(vimeo_root: Path, sequence_ids: list, seed: int, test_fraction: float = 0.1) -> None:
    '''(Re)write sep_trainlist.txt / sep_testlist.txt scoped to exactly the
    sequence ids available from the chunk currently linked into
    vimeo_root/sequences. nvc.data.vimeo treats these two files as
    authoritative and never re-splits them itself - something has to
    produce them, and this always overwrites the previous chunk's lists
    (correct here, since sequences/ was just replaced too).

    This is a per-chunk *training-progress* split, not a persistent
    benchmark - it is discarded with the chunk. It does NOT reproduce the
    official Vimeo-90K train/test split (that requires every chunk merged
    at once). Use DAVIS's existing test split for the actual final
    comparison between models trained by different people.
    '''
    ordered = sorted(sequence_ids)
    shuffled = ordered.copy()
    random.Random(seed).shuffle(shuffled)
    n_test = max(1, int(len(shuffled) * test_fraction))
    test_ids = sorted(shuffled[:n_test])
    train_ids = sorted(shuffled[n_test:])

    (vimeo_root / 'sep_trainlist.txt').write_text('\n'.join(train_ids) + '\n', encoding='utf-8')
    (vimeo_root / 'sep_testlist.txt').write_text('\n'.join(test_ids) + '\n', encoding='utf-8')
    print(f'[split] {len(train_ids)} train / {len(test_ids)} test sequence ids for this chunk')


def build_chunk_manifests(vimeo_root: Path, seed: int):
    train_path = defaults.vimeo_manifest_path.with_stem(defaults.vimeo_manifest_path.stem + '_train')
    test_path = defaults.vimeo_manifest_path.with_stem(defaults.vimeo_manifest_path.stem + '_test')
    build_sequence_manifest(vimeo_root, train_path, split='train', max_sequences=None, seed=seed, validate=True)
    build_sequence_manifest(vimeo_root, test_path, split='test', max_sequences=None, seed=seed, validate=True)
    return train_path, test_path


def load_progress() -> dict:
    if PROGRESS_PATH.is_file():
        return json.loads(PROGRESS_PATH.read_text(encoding='utf-8'))
    return {'completed_chunks': [], 'best_val_loss': None}


def save_progress(progress: dict) -> None:
    PROGRESS_PATH.write_text(json.dumps(progress, indent=2), encoding='utf-8')

## 5b. Milestone 8A: generate the QAT training-noise calibration

Only runs when `QAT_ENABLED` is True, and only the first time (skipped once
`QAT_CALIBRATION_PATH` already exists on Drive, so re-running after a
disconnect doesn't redo it). Downloads one small Vimeo chunk, computes fixed
per-channel quantization parameters from its TRAIN split against
`QAT_RESUME_FROM` (the checkpoint about to be fine-tuned), and writes the
result to Drive - the exact mechanism `scripts/calibrate_quantizer.py` uses
locally, reusing this notebook's own chunk-download helpers since Vimeo data
only exists transiently here, not as a persistent local manifest.


In [ ]:
if QAT_ENABLED and not QAT_CALIBRATION_PATH.is_file():
    import numpy as np

    from nvc.compression import (
        EmpiricalEntropyModel,
        calibrate_quantization_params,
        collect_calibration_latents,
        latent_to_symbols,
        save_calibration,
    )
    from nvc.training import load_model_from_checkpoint

    if not QAT_RESUME_FROM.is_file():
        raise FileNotFoundError(
            f'QAT_RESUME_FROM ({QAT_RESUME_FROM}) does not exist - the baseline Vimeo '
            'checkpoint must exist before the QAT calibration/training run can start.'
        )

    print(f'[QAT calibration] {QAT_CALIBRATION_PATH} not found - generating it from Vimeo TRAIN data...')
    calibration_model, _ = load_model_from_checkpoint(QAT_RESUME_FROM, device=device)

    calibration_chunk_root = download_and_extract_chunk(CHUNK_NUMBERS[0], LOCAL_SCRATCH_DIR)
    relink_sequences_to_chunk(calibration_chunk_root)
    calibration_sequence_ids = discover_complete_sequence_ids(VIMEO_ROOT / 'sequences')
    if not calibration_sequence_ids:
        raise RuntimeError('no complete sequences found in the calibration chunk')
    write_chunk_split_lists(VIMEO_ROOT, calibration_sequence_ids, seed=SEED)
    calibration_train_manifest, _ = build_chunk_manifests(VIMEO_ROOT, seed=SEED)

    calibration_loader = create_sequence_train_loader(
        calibration_train_manifest, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
        seed=SEED, crop_size=CROP_SIZE,
    )

    print('[QAT calibration] collecting latents from the TRAIN split...')
    calibration_latents = collect_calibration_latents(
        calibration_model, calibration_loader, device, max_batches=50,
    )
    calibration_params = calibrate_quantization_params(
        calibration_latents, bits=QAT_BITS, mode=QAT_MODE,
    )

    calibration_symbols = np.stack([
        latent_to_symbols(calibration_latents[i:i + 1], calibration_params).reshape(calibration_latents.shape[1:])
        for i in range(calibration_latents.shape[0])
    ])
    calibration_entropy_model = EmpiricalEntropyModel.from_symbols(
        calibration_symbols, bits=QAT_BITS, num_tables=calibration_latents.shape[1],
    )

    save_calibration(
        QAT_CALIBRATION_PATH, params=calibration_params,
        entropy_model_data=calibration_entropy_model.to_dict(),
        metadata={
            'method': 'per_channel_percentile', 'bits': QAT_BITS, 'mode': QAT_MODE,
            'calibration_frames': int(calibration_latents.shape[0]), 'calibration_split': 'train',
            'checkpoint': str(QAT_RESUME_FROM), 'seed': SEED,
            'purpose': 'Milestone 8A QAT training-time noise scale (not the deployed codec calibration)',
        },
    )
    print(f'[QAT calibration] wrote {QAT_CALIBRATION_PATH} from {calibration_latents.shape[0]} calibration frames')

    if LOCAL_SCRATCH_DIR.exists():
        shutil.rmtree(LOCAL_SCRATCH_DIR)
elif QAT_ENABLED:
    print(f'[QAT calibration] reusing existing {QAT_CALIBRATION_PATH}')


## 6. Model, optimizer, resume from the last checkpoint if one exists

In [ ]:
quantization_noise = None
if QAT_ENABLED:
    quantization_noise = QuantizationNoise.from_calibration(QAT_CALIBRATION_PATH, bits=QAT_BITS, mode=QAT_MODE)
    print(f'[QAT] Quantization-noise relaxation ENABLED - {quantization_noise.bits}-bit / '
          f'{quantization_noise.mode}, scale from {QAT_CALIBRATION_PATH} (distortion-only: loss is plain MSE).')
elif QAT_CONTROL_RUN:
    print('[QAT] Control run: plain-MSE fine-tune from QAT_RESUME_FROM, matched epoch budget, no noise.')
else:
    print('[QAT] Quantization-noise relaxation disabled (baseline training).')

model = BaselineAutoencoder(latent_channels=latent_channels, quantization_noise=quantization_noise).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
model_config = model.config_dict()

latest_ckpt = CHECKPOINT_DIR / 'latest.pt'
history = []
if latest_ckpt.is_file():
    epoch, history = resume_training_state(latest_ckpt, model=model, optimizer=optimizer, map_location=device)
    print(f'[resume] Loaded {latest_ckpt} - continuing from epoch {epoch}, {len(history)} epoch(s) of prior history')
elif (QAT_ENABLED or QAT_CONTROL_RUN) and QAT_RESUME_FROM.is_file():
    # First run of this experiment: fine-tune from the existing non-QAT
    # checkpoint rather than random init - same bootstrap source for QAT
    # and its control, so the two are only different in noise injection,
    # not starting weights. Epoch numbering restarts at 1 here - this is a
    # new, separately-tracked experiment (its own CHECKPOINT_DIR/history),
    # not a continuation of QAT_RESUME_FROM's own epoch count.
    resume_training_state(QAT_RESUME_FROM, model=model, optimizer=optimizer, map_location=device)
    epoch = 1
    print(f'[bootstrap] Loaded weights from {QAT_RESUME_FROM} to start this experiment fresh at epoch 1')
else:
    epoch = 1
    print('[resume] No existing checkpoint on Drive - starting fresh')

progress = load_progress()
best_val_loss = progress['best_val_loss'] if progress['best_val_loss'] is not None else float('inf')
completed_so_far = progress['completed_chunks']
print(f'Already-completed chunks: {completed_so_far}')


## 7. Main loop: download -> train -> checkpoint -> delete -> next chunk

Safe to re-run after a disconnect - already-completed chunks are skipped, and training continues from the last saved checkpoint.

In [ ]:
for chunk_number in CHUNK_NUMBERS:
    if chunk_number in progress['completed_chunks']:
        print(f'[chunk {chunk_number}] already completed - skipping')
        continue

    sep = '=' * 60
    print()
    print(sep)
    print(f'[chunk {chunk_number}] starting')
    print(sep)

    try:
        chunk_group_root = download_and_extract_chunk(chunk_number, LOCAL_SCRATCH_DIR)
        group_names = relink_sequences_to_chunk(chunk_group_root)
        print(f'[chunk {chunk_number}] linked {len(group_names)} group folder(s)')

        sequence_ids = discover_complete_sequence_ids(VIMEO_ROOT / 'sequences')
        print(f'[chunk {chunk_number}] {len(sequence_ids)} complete 7-frame sequences found')
        if not sequence_ids:
            raise RuntimeError('no complete sequences found in this chunk')

        write_chunk_split_lists(VIMEO_ROOT, sequence_ids, seed=SEED)
        train_manifest, test_manifest = build_chunk_manifests(VIMEO_ROOT, seed=SEED)

        train_loader = create_sequence_train_loader(
            train_manifest, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, seed=SEED, crop_size=CROP_SIZE,
        )
        test_loader = create_sequence_test_loader(
            test_manifest, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, crop_size=CROP_SIZE,
        )

        for _ in range(EPOCHS_PER_CHUNK):
            train_metrics = train_one_epoch(model, train_loader, optimizer, device)
            val_metrics = validate_one_epoch(model, test_loader, device)

            train_loss = train_metrics['loss']
            val_loss = val_metrics['loss']
            val_psnr = val_metrics['psnr']

            history.append({
                'epoch': epoch, 'chunk': chunk_number,
                'train_loss': train_loss, 'val_loss': val_loss, 'val_psnr': val_psnr,
                'run_type': RUN_TYPE,
                'qat_enabled': QAT_ENABLED,
                'qat_bits': QAT_BITS if QAT_ENABLED else None,
                'qat_mode': QAT_MODE if QAT_ENABLED else None,
            })
            print(f'[chunk {chunk_number}] epoch {epoch}: train_mse={train_loss:.6f} val_mse={val_loss:.6f} val_psnr={val_psnr:.2f} dB')

            save_checkpoint(CHECKPOINT_DIR / 'latest.pt', model=model, optimizer=optimizer,
                             epoch=epoch, history=history, model_config=model_config)
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                best_path = CHECKPOINT_DIR / 'best.pt'
                save_checkpoint(best_path, model=model, optimizer=optimizer,
                                 epoch=epoch, history=history, model_config=model_config)
                print(f'  [best] new best val MSE {best_val_loss:.6f} -> {best_path}')
            (CHECKPOINT_DIR / 'history.json').write_text(json.dumps(history, indent=2), encoding='utf-8')

            epoch += 1

        progress['completed_chunks'].append(chunk_number)
        progress['best_val_loss'] = best_val_loss
        save_progress(progress)
        print(f'[chunk {chunk_number}] done, progress saved')

    except Exception as exc:
        print(f'[chunk {chunk_number}] FAILED: {exc!r} - not marked complete, re-run this cell to retry it')
    finally:
        if DELETE_CHUNK_AFTER_TRAINING and LOCAL_SCRATCH_DIR.exists():
            shutil.rmtree(LOCAL_SCRATCH_DIR)

print()
print('All requested chunks processed (or already were).')

## 8. Download the trained model to your laptop

Run this any time you want a copy - it does not need the loop above to be fully finished.

In [ ]:
from google.colab import files

print('Checkpoints on Drive:')
for p in sorted(CHECKPOINT_DIR.glob('*.pt')):
    size_mb = p.stat().st_size / 1e6
    print(' ', p, f'({size_mb:.1f} MB)')

files.download(str(CHECKPOINT_DIR / 'best.pt'))

## 9. Milestone 8A (combined run): QAT + control together, per-chunk early stopping

An alternative to Sections 6-8 above, not a replacement - those still work
for training one model at a time. This section trains **both** the QAT and
control models in the same script: each Vimeo chunk is downloaded **once**
and both models train on it before it's deleted (no duplicate downloads),
and each model gets up to `EPOCHS_PER_CHUNK_MAX` epochs per chunk with its
**own** early stopping - a model that plateaus on a chunk stops early and
simply waits for the next chunk while the other model keeps going.

**Before running this section**, if `checkpoints_qat_noise/` or
`checkpoints_qat_control/` already have partial progress on Drive that you
want to discard and restart from `vimeo_epoch17_best.pt` at epoch 1 for
both models (e.g. you're switching from Section 7's approach to this one),
delete those two folders and `progress_qat_noise.json` /
`progress_qat_control.json` from `neural_streaming_colab/` on Drive first -
otherwise this section resumes whatever is already there, same as Section 7
does.

Requires `QAT_CALIBRATION_PATH` to already exist (Section 5b above, run
once with `QAT_ENABLED = True`).

In [ ]:
EPOCHS_PER_CHUNK_MAX = 10    # ceiling per chunk; early stopping usually stops sooner
EARLY_STOP_PATIENCE = 2      # consecutive non-improving epochs (within one chunk) before stopping early
EARLY_STOP_MIN_DELTA = 1e-5  # minimum val_loss improvement required to reset patience


def _load_or_init_progress(progress_path):
    if progress_path.is_file():
        return json.loads(progress_path.read_text(encoding='utf-8'))
    return {'completed_chunks': [], 'best_val_loss': None}


def _save_progress(progress_path, progress):
    progress_path.write_text(json.dumps(progress, indent=2), encoding='utf-8')


def _bootstrap_run(run_name, model, optimizer, checkpoint_dir, bootstrap_from):
    '''Resume this run's own latest.pt if present, else bootstrap weights
    from `bootstrap_from` and restart epoch numbering at 1, else random init.
    Mirrors Section 6's logic, factored out so both models can share it.'''
    latest_ckpt = checkpoint_dir / 'latest.pt'
    if latest_ckpt.is_file():
        epoch, history = resume_training_state(latest_ckpt, model=model, optimizer=optimizer, map_location=device)
        print(f'[{run_name}] resumed from {latest_ckpt} - next epoch {epoch}, {len(history)} prior record(s)')
        return epoch, history
    if bootstrap_from is not None and bootstrap_from.is_file():
        resume_training_state(bootstrap_from, model=model, optimizer=optimizer, map_location=device)
        print(f'[{run_name}] bootstrapped from {bootstrap_from}, starting fresh at epoch 1')
        return 1, []
    print(f'[{run_name}] no existing checkpoint - starting from random init at epoch 1')
    return 1, []


def _train_one_chunk_with_early_stopping(
    run_name, model, optimizer, train_loader, test_loader, *, start_epoch,
    max_epochs, patience, min_delta, chunk_number, history, checkpoint_dir,
    model_config, best_val_loss, run_type, qat_enabled,
):
    '''Up to max_epochs epochs on one chunk, stopping early if val_loss
    hasn't improved by min_delta for `patience` consecutive epochs.
    Returns (next_epoch, updated_best_val_loss). best_val_loss/checkpointing
    is GLOBAL (across all chunks), matching Section 7's semantics - only the
    early-stopping decision itself is scoped to this one chunk.'''
    epoch = start_epoch
    best_chunk_val_loss = float('inf')
    epochs_without_improvement = 0

    for step in range(max_epochs):
        train_metrics = train_one_epoch(model, train_loader, optimizer, device)
        val_metrics = validate_one_epoch(model, test_loader, device)
        train_loss, val_loss, val_psnr = train_metrics['loss'], val_metrics['loss'], val_metrics['psnr']

        history.append({
            'epoch': epoch, 'chunk': chunk_number,
            'train_loss': train_loss, 'val_loss': val_loss, 'val_psnr': val_psnr,
            'run_type': run_type, 'qat_enabled': qat_enabled,
            'qat_bits': QAT_BITS if qat_enabled else None,
            'qat_mode': QAT_MODE if qat_enabled else None,
        })
        print(f'  [{run_name}] chunk {chunk_number} epoch {epoch} (step {step + 1}/{max_epochs}): '
              f'train_mse={train_loss:.6f} val_mse={val_loss:.6f} val_psnr={val_psnr:.2f} dB')

        save_checkpoint(checkpoint_dir / 'latest.pt', model=model, optimizer=optimizer,
                         epoch=epoch, history=history, model_config=model_config)
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            save_checkpoint(checkpoint_dir / 'best.pt', model=model, optimizer=optimizer,
                             epoch=epoch, history=history, model_config=model_config)
            print(f'    [{run_name}] new global-best val MSE {best_val_loss:.6f} -> {checkpoint_dir / "best.pt"}')
        (checkpoint_dir / 'history.json').write_text(json.dumps(history, indent=2), encoding='utf-8')

        epoch += 1
        if val_loss < best_chunk_val_loss - min_delta:
            best_chunk_val_loss = val_loss
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= patience:
                print(f'  [{run_name}] early stop on chunk {chunk_number} - no improvement for '
                      f'{patience} epoch(s) (used {step + 1}/{max_epochs})')
                break

    return epoch, best_val_loss


In [ ]:
if not QAT_CALIBRATION_PATH.is_file():
    raise FileNotFoundError(
        f'{QAT_CALIBRATION_PATH} not found - run the Section 5b calibration cell '
        '(with QAT_ENABLED = True) before this combined-training section.'
    )
combined_quantization_noise = QuantizationNoise.from_calibration(QAT_CALIBRATION_PATH, bits=QAT_BITS, mode=QAT_MODE)
print(f'[qat] noise scale loaded - {combined_quantization_noise.bits}-bit / {combined_quantization_noise.mode}')

runs = {}
for run_type, noise in (('qat', combined_quantization_noise), ('qat_control', None)):
    checkpoint_dir = DRIVE_PROJECT_DIR / ('checkpoints_qat_noise' if run_type == 'qat' else 'checkpoints_qat_control')
    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    progress_path = DRIVE_PROJECT_DIR / ('progress_qat_noise.json' if run_type == 'qat' else 'progress_qat_control.json')

    run_model = BaselineAutoencoder(latent_channels=latent_channels, quantization_noise=noise).to(device)
    run_optimizer = torch.optim.Adam(run_model.parameters(), lr=learning_rate)
    run_epoch, run_history = _bootstrap_run(run_type, run_model, run_optimizer, checkpoint_dir, QAT_RESUME_FROM)
    run_progress = _load_or_init_progress(progress_path)

    runs[run_type] = {
        'model': run_model, 'optimizer': run_optimizer, 'model_config': run_model.config_dict(),
        'checkpoint_dir': checkpoint_dir, 'progress_path': progress_path,
        'epoch': run_epoch, 'history': run_history, 'progress': run_progress,
        'best_val_loss': run_progress['best_val_loss'] if run_progress['best_val_loss'] is not None else float('inf'),
        'qat_enabled': run_type == 'qat',
    }

for chunk_number in CHUNK_NUMBERS:
    pending = [name for name, run in runs.items() if chunk_number not in run['progress']['completed_chunks']]
    if not pending:
        print(f'[chunk {chunk_number}] already completed by both runs - skipping')
        continue

    sep = '=' * 60
    print()
    print(sep)
    print(f'[chunk {chunk_number}] starting (pending: {pending})')
    print(sep)

    try:
        chunk_group_root = download_and_extract_chunk(chunk_number, LOCAL_SCRATCH_DIR)
        group_names = relink_sequences_to_chunk(chunk_group_root)
        print(f'[chunk {chunk_number}] linked {len(group_names)} group folder(s)')

        sequence_ids = discover_complete_sequence_ids(VIMEO_ROOT / 'sequences')
        print(f'[chunk {chunk_number}] {len(sequence_ids)} complete 7-frame sequences found')
        if not sequence_ids:
            raise RuntimeError('no complete sequences found in this chunk')

        write_chunk_split_lists(VIMEO_ROOT, sequence_ids, seed=SEED)
        train_manifest, test_manifest = build_chunk_manifests(VIMEO_ROOT, seed=SEED)

        # Built once, shared by both models - this is the actual saving
        # versus running Sections 6-8 twice (each would re-download the
        # same chunk independently).
        train_loader = create_sequence_train_loader(
            train_manifest, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, seed=SEED, crop_size=CROP_SIZE,
        )
        test_loader = create_sequence_test_loader(
            test_manifest, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, crop_size=CROP_SIZE,
        )

        for name in pending:
            run = runs[name]
            run['epoch'], run['best_val_loss'] = _train_one_chunk_with_early_stopping(
                name, run['model'], run['optimizer'], train_loader, test_loader,
                start_epoch=run['epoch'], max_epochs=EPOCHS_PER_CHUNK_MAX,
                patience=EARLY_STOP_PATIENCE, min_delta=EARLY_STOP_MIN_DELTA,
                chunk_number=chunk_number, history=run['history'],
                checkpoint_dir=run['checkpoint_dir'], model_config=run['model_config'],
                best_val_loss=run['best_val_loss'], run_type=name, qat_enabled=run['qat_enabled'],
            )
            run['progress']['completed_chunks'].append(chunk_number)
            run['progress']['best_val_loss'] = run['best_val_loss']
            _save_progress(run['progress_path'], run['progress'])
            print(f'[chunk {chunk_number}] [{name}] done, progress saved')

    except Exception as exc:
        print(f'[chunk {chunk_number}] FAILED: {exc!r} - not marked complete, re-run this cell to retry it')
    finally:
        if DELETE_CHUNK_AFTER_TRAINING and LOCAL_SCRATCH_DIR.exists():
            shutil.rmtree(LOCAL_SCRATCH_DIR)

print()
print('All requested chunks processed (or already were) for both runs.')


## 10. Download both trained models

Run this any time - it does not need the loop above to be fully finished.

In [ ]:
from google.colab import files

for run_type in ('qat', 'qat_control'):
    checkpoint_dir = DRIVE_PROJECT_DIR / ('checkpoints_qat_noise' if run_type == 'qat' else 'checkpoints_qat_control')
    print(f'[{run_type}] checkpoints on Drive:')
    for p in sorted(checkpoint_dir.glob('*.pt')):
        size_mb = p.stat().st_size / 1e6
        print(' ', p, f'({size_mb:.1f} MB)')

files.download(str(DRIVE_PROJECT_DIR / 'checkpoints_qat_noise' / 'best.pt'))
files.download(str(DRIVE_PROJECT_DIR / 'checkpoints_qat_control' / 'best.pt'))
